In [1]:
import pickle, numpy as np, pandas as pd
from sklearn.cluster import AgglomerativeClustering

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import os
from huggingface_hub import InferenceClient

C:\Users\whdgu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#read in complaint embeddings
embed_dict = pickle.load(open('../data/complaint_embeddings.pkl', 'rb'))
#sort by complaint id
sorted_indices = np.argsort(embed_dict['complaint_id'])
embeddings = embed_dict['embeddings'][sorted_indices]
#create df with complaint_id, product, subproduct, company
embeddings_df = pd.DataFrame({
    'complaint_id': np.array(embed_dict['complaint_id'])[sorted_indices],
    'product': np.array(embed_dict['product'])[sorted_indices],
    'subproduct': np.array(embed_dict['sub-product'])[sorted_indices],
    'company': np.array(embed_dict['company'])[sorted_indices],
    'narrative': np.array(embed_dict['narrative'])[sorted_indices],
    'embeddings': list(embeddings)
})
embeddings_df.head(2)

,complaint_id,product,subproduct,company,narrative,embeddings
0,10645738,"Money transfer, virtual currency, or money ser...",Check cashing service,"Paypal Holdings, Inc",I want to withdraw my funds from my Venmo acco...,"[-0.20069551, -0.072827876, 0.0846661, -0.2364..."
1,10647276,Checking or savings account,Savings account,"Block, Inc.",I saw that cashapp recently opened a savings a...,"[-0.04179181, -0.019972146, 0.13940147, -0.038..."


##helper functions

In [3]:

#get embeddings for a specific complaint
def get_embeddings(complaint_id):
    complaint_indices = [i for i, c in enumerate(embed_dict['complaint_id']) if c == complaint_id]
    complaint__indices = sorted(complaint_indices)
    embeddings = embed_dict['embeddings'][complaint_indices]
    return embeddings

def get_experiment_scores(embeddings, cluster_labels):
    '''Get the silhouette score, calinski_harabasz_score, davies_bouldin_score for the given embeddings and distance threshold
    Args:
        embeddings (np.array): The embeddings to cluster.
        cluster_labels (list): The cluster labels for the embeddings.
    Returns:
        dict: A dictionary with the silhouette score, calinski_harabasz_score, davies_bouldin_score.

        silohouette score ranges from -1 to 1, with 1 being the best score.
        calinski_harabasz_score ranges from 0 to infinity, with higher values indicating better clustering.
        davies_bouldin_score ranges from 0 to infinity, with lower values indicating better clustering.
    
    '''
    #get the silhouette score, calinski_harabasz_score, davies_bouldin_score for the given embeddings and distance threshold
    if len(set(cluster_labels)) == 1:
        return {'silhouette': -1,
                'calinski_harabasz': 0,
                'davies_bouldin': float('inf')}
                
    silhouette = silhouette_score(embeddings, cluster_labels)
    calinski_harabasz = calinski_harabasz_score(embeddings, cluster_labels)
    davies_bouldin = davies_bouldin_score(embeddings, cluster_labels)

    return {'silhouette': silhouette,
            'calinski_harabasz': calinski_harabasz,
            'davies_bouldin': davies_bouldin}

def cluster_embeddings(embeeding_df, product, distance_threshold=[1, 5, 10, 15, 20, 25, 30, 35, 40]):
    ''' Cluster the embeddings for a specific product using agglomerative clustering
    Args:

        embeeding_df (pd.DataFrame): The dataframe with embeddings and complaint metadata.
        product (str): The product to cluster.
        distance_threshold (list): The distance thresholds to use for clustering.
    Returns:
        cluster_results (pd.DataFrame): A dataframe with the clustering results for each distance threshold.
    '''
    product_df = embeddings_df.loc[embeddings_df['product'] == product, :]
    if product_df.shape[0] < 2:
        print (f"Not enough complaints for product {product} to cluster.")
        return None, None
    embeddings = np.vstack(embeddings_df.loc[embeddings_df['product'] == product, 'embeddings'].to_numpy())
    cluster_results = pd.DataFrame()
    for dt in distance_threshold:
        clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=dt)
        cluster_labels = clustering.fit_predict(embeddings)
        #write cluster assignments to dataframe with coulmn name 'cluster_label_{dt}'
        product_df[f'cluster_label_{dt}'] = cluster_labels
        num_clusters = len(set(cluster_labels))
        scores = get_experiment_scores(embeddings, cluster_labels)
        avg_cluster_size = len(embeddings) / num_clusters
        cluster_results = pd.concat([cluster_results, pd.DataFrame({
            'distance_threshold': [dt], 
            'num_clusters': [num_clusters],
            'avg_cluster_size': [avg_cluster_size],
            'silhouette': [scores['silhouette']],
            'calinski_harabasz': [scores['calinski_harabasz']],
            'davies_bouldin': [scores['davies_bouldin']]
        })], ignore_index=True)
                                
        print (f"Distance threshold: {dt}, Number of clusters: {num_clusters}")
    return cluster_results, product_df

def cluster_with_gpu(embeddings_subset, distance_threshold):
    ''' Cluster the embeddings with GPU using dbscan    '''
    

In [4]:
#size by product
embeddings_df['product'].value_counts()

product
Money transfer, virtual currency, or money service         54312
Checking or savings account                                31893
Credit card                                                21338
Credit reporting or other personal consumer reports        15246
Debt collection                                             4404
Prepaid card                                                1751
Vehicle loan or lease                                       1669
Mortgage                                                    1243
Payday loan, title loan, personal loan, or advance loan      415
Debt or credit management                                    205
Student loan                                                  36
Name: count, dtype: int64

In [ ]:
#try clustering each product's embeddings separately

products = embeddings_df['product'].unique()
product_summary = pd.DataFrame()
embeddings_df_clustered = pd.DataFrame()
for product in products:
    print(f"Clustering product: {product}")
    cluster_results, product_df = cluster_embeddings(embeddings_df, product, distance_threshold=[5, 10, 15, 20, 25, 30, 35, 40])
    embeddings_df_clustered = pd.concat([embeddings_df_clustered, product_df], ignore_index=True)
    #print(f'cluster results for product {product}:')
    #print(cluster_results)
    cluster_results['product'] = product
    product_summary = pd.concat([product_summary, cluster_results], ignore_index=True)
product_summary.to_csv('../data/product_clustering_summary.csv', index=False)
product_summary


    

Clustering product: Money transfer, virtual currency, or money service


C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels


Distance threshold: 5, Number of clusters: 818


C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels


Distance threshold: 10, Number of clusters: 174


...

C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

Distance threshold: 40, Number of clusters: 2
cluster results for product Mortgage:
   distance_threshold  num_clusters  avg_cluster_size  silhouette  \
0                   5            58         21.431034    0.019364   
1                  10            10        124.300000    0.057200   
2                  15             6        207.166667    0.053465   
3                  20             3        414.333333    0.194387   
4                  25             3        414.333333    0.194387   
5                  30             3        414.333333    0.194387   
6                  35             3        414.333333    0.194387   
7                  40             2        621.500000    0.252239   

   calinski_harabasz  davies_bouldin  
0          29.806292        2.617591  
1         124.541687        3.629996  
2         196.749268        3.027958  
3         399.216644        2.100187  
4         399.216644        2.100187  
5         399.216644        2.100187  
6         399.216644 

C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_df[f'cluster_label_{dt}'] = cluster_labels
C:\Users\whdgu\AppData\Local\Temp\ipykernel_11440\634012507.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

Distance threshold: 5, Number of clusters: 6
Distance threshold: 10, Number of clusters: 2
Distance threshold: 15, Number of clusters: 2
Distance threshold: 20, Number of clusters: 1
Distance threshold: 25, Number of clusters: 1
Distance threshold: 30, Number of clusters: 1
Distance threshold: 35, Number of clusters: 1
Distance threshold: 40, Number of clusters: 1
cluster results for product Student loan:
   distance_threshold  num_clusters  avg_cluster_size  silhouette  \
0                   5             6               6.0    0.238378   
1                  10             2              18.0    0.342507   
2                  15             2              18.0    0.342507   
3                  20             1              36.0   -1.000000   
4                  25             1              36.0   -1.000000   
5                  30             1              36.0   -1.000000   
6                  35             1              36.0   -1.000000   
7                  40             1    

,distance_threshold,num_clusters,avg_cluster_size,silhouette,calinski_harabasz,davies_bouldin,product
0,5,818,66.396088,0.308903,255.075989,2.591984,"Money transfer, virtual currency, or money ser..."
1,10,174,312.137931,0.291343,879.757324,3.021499,"Money transfer, virtual currency, or money ser..."
2,15,74,733.945946,0.261424,1789.290527,3.083719,"Money transfer, virtual currency, or money ser..."
3,20,49,1108.408163,0.284425,2522.809326,2.924733,"Money transfer, virtual currency, or money ser..."
4,25,33,1645.818182,0.272828,3495.135742,2.775293,"Money transfer, virtual currency, or money ser..."
...,...,...,...,...,...,...,...
83,20,2,102.500000,0.165270,46.009750,2.056710,Debt or credit management
84,25,2,102.500000,0.165270,46.009750,2.056710,Debt or credit management
85,30,1,205.000000,-1.000000,0.000000,inf,Debt or credit management
86,35,1,205.000000,-1.000000,0.000000,inf,Debt or credit management


In [20]:
#get the best distance threshold for each product based on silhouette, calinski_harabasz, davies_bouldin with less than 50 clusters for each product
best_distance_thresholds = pd.DataFrame()
for product in products:
    product_df = product_summary.loc[product_summary['product'] == product, :]
    #filter to only distance thresholds with less than 50 clusters
    filtered_df = product_df.loc[product_df['num_clusters'] < 50, :]
    if filtered_df.shape[0] == 0:
        continue
    #get the distance threshold with the best silhouette score, calinski_harabasz, davies_bouldin
    best_silhouette = filtered_df.loc[filtered_df['silhouette'].idxmax(), :]
    best_calinski_harabasz = filtered_df.loc[filtered_df['calinski_harabasz'].idxmax(), :]
    best_davies_bouldin = filtered_df.loc[filtered_df['davies_bouldin'].idxmin(), :]
    best_distance_thresholds = pd.concat([best_distance_thresholds, pd.DataFrame({
        'product': [product],
        'best_distance_threshold': [best_silhouette['distance_threshold']],
        'num_clusters': [best_silhouette['num_clusters']],
        'silhouette': [best_silhouette['silhouette']],
        'calinski_harabasz': [best_calinski_harabasz['calinski_harabasz']],
        'davies_bouldin': [best_davies_bouldin['davies_bouldin']]
    })], ignore_index=True)
best_distance_thresholds.to_csv('outputbest_distance_thresholds.csv', index=False)
best_distance_thresholds


,product,best_distance_threshold,num_clusters,silhouette,calinski_harabasz,davies_bouldin
0,"Money transfer, virtual currency, or money ser...",40,16,0.415033,6352.220703,2.531879
1,Checking or savings account,30,26,0.183332,1926.806152,3.184537
2,Credit card,20,32,0.023643,1910.232422,3.947745
3,Credit reporting or other personal consumer re...,20,36,0.032476,1539.849609,3.116986
4,Debt collection,40,3,0.163849,1239.225586,2.331481
5,Vehicle loan or lease,25,3,0.170340,502.727295,2.256531
6,Prepaid card,40,3,0.145544,371.836243,2.141684
7,Mortgage,40,2,0.252239,555.106323,1.286305
8,"Payday loan, title loan, personal loan, or adv...",20,2,0.235300,168.921509,1.415371
9,Student loan,10,2,0.342507,24.502991,1.115325


In [6]:
embeddings_df_clustered

,complaint_id,product,subproduct,company,narrative,embeddings,cluster_label_5,cluster_label_10,cluster_label_15,cluster_label_20,cluster_label_25,cluster_label_30,cluster_label_35,cluster_label_40
0,10645738,"Money transfer, virtual currency, or money ser...",Check cashing service,"Paypal Holdings, Inc",I want to withdraw my funds from my Venmo acco...,"[-0.20069551, -0.072827876, 0.0846661, -0.2364...",526,4,39,9,19,19,19,1
1,10648930,"Money transfer, virtual currency, or money ser...",International money transfer,WELLS FARGO & COMPANY,"Dear CFPB Case Manager, I am writing to escala...","[-0.18225335, 0.05586333, 0.09726918, 0.097792...",514,47,8,17,4,1,1,4
2,10650395,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,"Paypal Holdings, Inc",My account was limited in XXXX and I was told ...,"[0.059066728, -0.29441375, 0.25489962, 0.07756...",311,2,6,14,30,0,5,1
3,10651988,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,"Paypal Holdings, Inc",Sometimes in XXXX I noticed XXXX credit cards ...,"[-0.16532695, 0.0060112597, -0.05945909, -0.02...",87,47,8,17,4,1,1,4
4,10652510,"Money transfer, virtual currency, or money ser...",International money transfer,"BANK OF AMERICA, NATIONAL ASSOCIATION","On date XX/XX/year> I received a wire, on hist...","[-0.037552837, 0.016087072, 0.25978428, 0.1511...",774,48,11,21,22,22,2,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132507,15431915,Debt or credit management,Credit repair services,WELLS FARGO & COMPANY,I applied for a personal loan to repay my cred...,"[-0.2564726, -0.1275893, 0.22124344, 0.0722324...",4,4,0,1,1,0,0,0
132508,15481146,Debt or credit management,Credit repair services,"BANK OF AMERICA, NATIONAL ASSOCIATION",My ex husband is using a credit card that is X...,"[-0.10727175, -0.19191548, 0.28368923, -0.1621...",14,4,0,1,1,0,0,0
132509,15615130,Debt or credit management,Debt settlement,"CITIBANK, N.A.","To the Consumer Financial Protection Bureau, I...","[-0.2940723, 0.110055685, 0.020373518, 0.01175...",8,2,2,0,0,0,0,0
132510,15787998,Debt or credit management,Mortgage modification or foreclosure avoidance,WELLS FARGO & COMPANY,Wells Fargo and Hud Servicer XXXX have a withh...,"[-0.330733, 0.042372834, 0.14269748, 0.0155245...",15,2,2,0,0,0,0,0


In [15]:
def get_llm_response(prompt):
    client = InferenceClient()

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )
    return completion.choices[0].message['content']
def get_llm__cluster_name(cluster_narratives):
    prompt = f"Given the following customer complaint narratives, provide a concise name that captures the common theme among them. Do not include a company name in your theme name. Narratives: {cluster_narratives}"
    response = get_llm_response(prompt)
    return response

def name_clusters(embeddings_df, cluster_col='cluster_label',product_subset=None):
    '''
    Name clusters using LLM based on a smaple of narratives from each each product and cluster label.
    Args:
        embeddings_df (pd.DataFrame): DataFrame containing complaint narratives and cluster labels.
        cluster_col (str): Column name in embeddings_df that contains cluster labels.
    Returns:
        dict: A dictionary mapping products and cluster labels to their assigned names.

    '''
    #check that there are less than 50 clusters



    cluster_names = pd.DataFrame(columns=['product','cluster_column', 'cluster_label', 'cluster_name'])
    if product_subset is not None:
        embeddings_df = embeddings_df.loc[embeddings_df['product'].isin(product_subset), :]
        products = product_subset
    else:
        products = embeddings_df['product'].unique()
    for product in products:
        product_df = embeddings_df.loc[embeddings_df['product'] == product, :]
        cluster_labels = product_df[cluster_col].unique()
        if len(cluster_labels) > 50:
            print (f"Skipping product {product} with {len(cluster_labels)} clusters.")
            continue
        for cluster_label in cluster_labels:
            cluster_df = product_df.loc[product_df[cluster_col] == cluster_label, :]
            #sample up to 5 narratives from the cluster
            sample_narratives = cluster_df['narrative'].sample(n=min( len(cluster_df), 7), random_state=42).tolist()
            cluster_name = get_llm__cluster_name(sample_narratives)
            cluster_names = pd.concat([cluster_names, pd.DataFrame({'product': product,'cluster_column': cluster_col, 'cluster_label': cluster_label, 'cluster_name': cluster_name}, index=[0])], ignore_index=True)
            print(f"Product: {product},cluster Column: {cluster_col}, Cluster: {cluster_label}, Name: {cluster_name}")
    return cluster_names


#add cluster names to embeddings_df


In [16]:
cluster_names_df = pd.DataFrame()
for product in products:
    #run over columns with cluster in the name
    cluster_cols = [col for col in embeddings_df_clustered.columns if 'cluster' in col]
    for cluster_col in cluster_cols:
        print(f"Product: {product}, Cluster Column: {cluster_col}")
        cluster_names = name_clusters(embeddings_df_clustered, cluster_col=cluster_col, product_subset=[product])
        cluster_names_df = pd.concat([cluster_names_df, cluster_names], ignore_index=True)


Product: Money transfer, virtual currency, or money service, Cluster Column: cluster_label_5
Skipping product Money transfer, virtual currency, or money service with 818 clusters.
Product: Money transfer, virtual currency, or money service, Cluster Column: cluster_label_10
Skipping product Money transfer, virtual currency, or money service with 174 clusters.
Product: Money transfer, virtual currency, or money service, Cluster Column: cluster_label_15
Skipping product Money transfer, virtual currency, or money service with 74 clusters.
Product: Money transfer, virtual currency, or money service, Cluster Column: cluster_label_20
Product: Money transfer, virtual currency, or money service,cluster Column: cluster_label_20, Cluster: 9, Name: **Frozen Account & Withheld Funds**
Product: Money transfer, virtual currency, or money service,cluster Column: cluster_label_20, Cluster: 17, Name: **Financial Fraud and Unresolved Consumer Loss**
Product: Money transfer, virtual currency, or money ser

In [18]:
cluster_names_df.to_csv('output/cluster_names.csv', index=False)
embeddings_df_clustered.to_csv('../data/embeddings_df_clustered.csv', index=False)
product_summary.to_csv('output/product_clustering_summary.csv', index=False)

In [ ]:
cluster_names